In [1]:
from mainnet_launch.pages.expected_vs_actual_returns.fetch_by_destination_realized_apr import (
    fetch_stuff,
    add_timestamp_to_df_with_block_column,
)
from mainnet_launch.constants import AUTO_ETH, BAL_ETH, WORKING_DATA_DIR


blocks, out_stats_df, in_stats_df, tvl_df, VaultLiquidated_df, last_snapshot_virtual_price_df, decay_state_df = (
    fetch_stuff(BAL_ETH, BAL_ETH.chain.block_autopool_first_deployed)
)

/home/parker/Documents/Tokemak/v2-rebalance-dashboard/mainnet_launch/pages/autopool_diagnostics/fetch_destination_summary_stats.py:187: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  merged_destination_df = pd.DataFrame.from_records(summary_stats_df.apply(_combine_to_destination_name, axis=1))
/home/parker/Documents/Tokemak/v2-rebalance-dashboard/mainnet_launch/pages/autopool_diagnostics/fetch_destination_summary_stats.py:187: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  merged_destination_df = pd.DataFrame.from_records(summary_stats_df.apply(_combine_to_destination_name, axis=1))


In [11]:
from mainnet_launch.pages.expected_vs_actual_returns.fetch_by_destination_realized_apr import (
    _build_onchain_expected_apr_components,
    extract_raw_onchain_returns_df,
    transform_concat_components,
    _set_base_apr_to_0_for_double_counting_destinations,
    transform_exclude_autopool_and_convert_decay,
    write_dataframe_to_table,
    drop_table,
)

autopool = BAL_ETH

expected_apr_df = _build_onchain_expected_apr_components(autopool, out_stats_df, in_stats_df)
raw_returns_df = extract_raw_onchain_returns_df(
    autopool, tvl_df, last_snapshot_virtual_price_df, VaultLiquidated_df, decay_state_df
)

combined_df = transform_concat_components(expected_apr_df, raw_returns_df)
combined_df = _set_base_apr_to_0_for_double_counting_destinations(combined_df)
combined_df = combined_df.reset_index()
combined_df = transform_exclude_autopool_and_convert_decay(combined_df, autopool)
combined_df["autopool"] = autopool.name

combined_df

index   baseApr  \
timestamp                 vault_name                                       
2024-09-15 00:00:00+00:00 B-rETH-STABLE (balancer)           0  0.581749   
                          ECLP-wstETH-cbETH (balancer)       1  1.362540   
                          ECLP-wstETH-wETH (balancer)        2  1.375177   
                          ETHx/wstETH (balancer)             3  1.429034   
                          balETH (tokemak)                   4  0.000000   
...                                                        ...       ...   
2025-03-02 00:00:00+00:00 rsETH / ETHx (balancer)         2192  0.000000   
                          rsETH / WETH (balancer)         2193  2.119171   
                          weETH/ezETH/rswETH (balancer)   2194  1.687830   
                          weETH/rETH (balancer)           2195  1.417591   
                          wstETH-WETH-BPT (balancer)      2196  0.000000   

                                                           feeApr  \
timestamp                 vault_name                                
2024-09-15 00:00:00+00:00 B-rETH-STABLE (balancer)       0.399744   
                          ECLP-wstETH-cbETH (balancer)   0.159975   
                          ECLP-wstETH-wETH (balancer)    0.540138   
                          ETHx/wstETH (balancer)         0.854648   
                          balETH (tokemak)               0.000000   
...                                                           ...   
2025-03-02 00:00:00+00:00 rsETH / ETHx (balancer)        0.000000   
                          rsETH / WETH (balancer)        0.784068   
                          weETH/ezETH/rswETH (balancer)  0.874893   
                          weETH/rETH (balancer)          0.305835   
                          wstETH-WETH-BPT (balancer)     0.000000   

                                                         incentiveAprOut  \
timestamp                 vault_name                                       
2024-09-15 00:00:00+00:00 B-rETH-STABLE (balancer)              4.123495   
                          ECLP-wstETH-cbETH (balancer)          3.561680   
                          ECLP-wstETH-wETH (balancer)           6.163647   
                          ETHx/wstETH (balancer)                9.791455   
                          balETH (tokemak)                      0.000000   
...                                                                  ...   
2025-03-02 00:00:00+00:00 rsETH / ETHx (balancer)               0.000000   
                          rsETH / WETH (balancer)               6.036005   
                          weETH/ezETH/rswETH (balancer)         6.212135   
                          weETH/rETH (balancer)                 6.073155   
                          wstETH-WETH-BPT (balancer)            0.000000   

                                                         incentiveAprIn  \
timestamp                 vault_name                                      
2024-09-15 00:00:00+00:00 B-rETH-STABLE (balancer)             4.123495   
                          ECLP-wstETH-cbETH (balancer)         3.561680   
                          ECLP-wstETH-wETH (balancer)          6.163647   
                          ETHx/wstETH (balancer)               9.791455   
                          balETH (tokemak)                     0.000000   
...                                                                 ...   
2025-03-02 00:00:00+00:00 rsETH / ETHx (balancer)              0.000000   
                          rsETH / WETH (balancer)              0.000000   
                          weETH/ezETH/rswETH (balancer)        5.099285   
                          weETH/rETH (balancer)                6.073155   
                          wstETH-WETH-BPT (balancer)           0.000000   

                                                                 TVL  \
timestamp                 vault_name                                   
2024-09-15 00:00:00+00:00 B-rETH-STABLE (balancer)          0

In [9]:
combined_df["autopool"] = autopool.name
combined_df = combined_df
drop_table("bad_df")
write_dataframe_to_table(combined_df, "bad_df")

Table 'bad_df' dropped successfully.
Table 'bad_df' created and data inserted.
Successfully updated table_name='bad_df' current_time=datetime.datetime(2025, 3, 3, 18, 25, 36, 880620, tzinfo=datetime.timezone.utc).


In [4]:
combined_df.reset_index()["vault_name"].value_counts()

vault_name
B-rETH-STABLE (balancer)         169
ECLP-wstETH-cbETH (balancer)     169
ECLP-wstETH-wETH (balancer)      169
ETHx/wstETH (balancer)           169
balETH (tokemak)                 169
ezETH-WETH-BPT (balancer)        169
osETH/wETH-BPT (balancer)        169
pxETH/wETH (balancer)            169
rsETH / ETHx (balancer)          169
rsETH / WETH (balancer)          169
weETH/ezETH/rswETH (balancer)    169
weETH/rETH (balancer)            169
wstETH-WETH-BPT (balancer)       169
Name: count, dtype: int64